In [1]:
here::i_am("snakemake/01_create_arrow.R")
source(here::here("settings.R"))

# I/O
io$output.directory <- file.path(io$basedir,"ArchR_test4")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final

Setting default number of Parallel threads to 1.



In [23]:
args = list()
args$sample = 'BGRGP1'
args$min_fragments = 100
args$min_tss_score = 0

In [59]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[150:200]
geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

Warning message in 1:150:200:
“numerical expression has 150 elements: only the first used”


In [60]:
# trim 2kb ends of geneAnnotation, otherwise gives error: 

exclude = GRanges(
    seqnames = Rle(rep(genomeAnnotation$chromSizes@seqnames@values,2)),
    ranges = IRanges(start = c(genomeAnnotation$chromSizes@ranges@start, 
                               genomeAnnotation$chromSizes@ranges@width-2000), 
                     end = c(genomeAnnotation$chromSizes@ranges@start + 2000, 
                             genomeAnnotation$chromSizes@ranges@width)))

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$TSS))
if(nrow(exclude_ranges)){
geneAnnotation$TSS = geneAnnotation$TSS[-exclude_ranges$subjectHits]
}

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$genes))
if(nrow(exclude_ranges)){
geneAnnotation$genes = geneAnnotation$genes[-exclude_ranges$subjectHits]
}
exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$exons))
if(nrow(exclude_ranges)){
geneAnnotation$exons = geneAnnotation$exons[-exclude_ranges$subjectHits]
}

In [61]:
geneAnnotation$TSS
geneAnnotation$genes
geneAnnotation$exons
genomeAnnotation$chromSizes 

GRanges object with 46272 ranges and 2 metadata columns:
           seqnames    ranges strand |     tx_id            tx_name
              <Rle> <IRanges>  <Rle> | <integer>        <character>
      [1]      chr1     15188      + |         1 ENSOCUT00000014253
      [2]      chr1     20324      + |         2 ENSOCUT00000038745
      [3]      chr1     52453      + |         3 ENSOCUT00000005053
      [4]      chr1     75200      + |         4 ENSOCUT00000005041
      [5]      chr1    125890      + |         5 ENSOCUT00000005032
      ...       ...       ...    ... .       ...                ...
  [46268] chrUn0178    325325      - |     47098 ENSOCUT00000034796
  [46269] chrUn0178    325307      - |     47099 ENSOCUT00000000866
  [46270] chrUn0178    324997      - |     47100 ENSOCUT00000063424
  [46271] chrUn0178    341912      - |     47101 ENSOCUT00000024657
  [46272] chrUn0178    476558      - |     47102 ENSOCUT00000014523
  -------
  seqinfo: 1402 sequences from an unspecified gen

GRanges object with 25912 ranges and 2 metadata columns:
           seqnames        ranges strand |            gene_id
              <Rle>     <IRanges>  <Rle> |        <character>
      [1]      chr1   15188-30379      + | ENSOCUG00000014251
      [2]      chr1   52453-53038      + | ENSOCUG00000005054
      [3]      chr1   57421-74906      - | ENSOCUG00000005046
      [4]      chr1   75200-85749      + | ENSOCUG00000005044
      [5]      chr1   92423-95846      - | ENSOCUG00000005040
      ...       ...           ...    ... .                ...
  [25908] chrUn0178 268869-270549      + | ENSOCUG00000033412
  [25909] chrUn0178 312451-325326      - | ENSOCUG00000000865
  [25910] chrUn0178 340582-341913      - | ENSOCUG00000028016
  [25911] chrUn0178 421791-423600      + | ENSOCUG00000031685
  [25912] chrUn0178 476220-476559      - | ENSOCUG00000021865
                      symbol
                 <character>
      [1]              WDR31
      [2]             RNF183
      [3]            

GRanges object with 213647 ranges and 3 metadata columns:
            seqnames        ranges strand |   exon_id            gene_id
               <Rle>     <IRanges>  <Rle> | <integer>        <character>
       [1]      chr1   15188-15282      + |         1 ENSOCUG00000014251
       [2]      chr1   20324-20326      + |         2 ENSOCUG00000014251
       [3]      chr1   20453-20603      + |         3 ENSOCUG00000014251
       [4]      chr1   20453-20603      + |         4 ENSOCUG00000014251
       [5]      chr1   21033-21163      + |         5 ENSOCUG00000014251
       ...       ...           ...    ... .       ...                ...
  [213643] chrUn0178 340582-340639      - |    215864 ENSOCUG00000028016
  [213644] chrUn0178 341454-341747      - |    215865 ENSOCUG00000028016
  [213645] chrUn0178 341783-341913      - |    215866 ENSOCUG00000028016
  [213646] chrUn0178 476220-476410      - |    215867 ENSOCUG00000021865
  [213647] chrUn0178 476521-476559      - |    215868 ENSOCUG00000

GRanges object with 200 ranges and 0 metadata columns:
         seqnames      ranges strand
            <Rle>   <IRanges>  <Rle>
    [1]      chr1 1-194850757      *
    [2]     chr10  1-47997241      *
    [3]     chr11  1-87554214      *
    [4]     chr12 1-155355395      *
    [5]     chr13 1-143360832      *
    ...       ...         ...    ...
  [196] chrUn0174    1-700076      *
  [197] chrUn0175    1-571908      *
  [198] chrUn0176    1-557607      *
  [199] chrUn0177    1-597847      *
  [200] chrUn0178    1-577984      *
  -------
  seqinfo: 3242 sequences from an unspecified genome

In [62]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)

ArchR logging to : ArchRLogs/ArchR-createArrows-4e0617e3a18-Date-2022-01-28_Time-15-03-47.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:03:48 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:03:48 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:03:48 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0.001 mins elapsed.

2022-01-28 15:03:48 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:07:04 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 1 Percent, 3.27 mins elapsed.



In [63]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [67]:
args$min_fragments = 2
args$min_tss_score = 0
for(i in 100:200){ 
    print(i)
    
    geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[i]
geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]


fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)
    }

[1] 100


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR logging to : ArchRLogs/ArchR-createArrows-4e061013f514-Date-2022-01-28_Time-15-09-30.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:09:30 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:09:31 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since not completed!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:09:31 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 15:09:31 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

2022-01-28 15:09:36 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 0.088 mins elapsed.

2022-01-28 15:09:36 : (rabbit_BGRGP1 : 1 of 1)

[1] 101


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR logging to : ArchRLogs/ArchR-createArrows-4e06d1f3cef-Date-2022-01-28_Time-15-09-58.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:09:59 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:09:59 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:09:59 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0.001 mins elapsed.

2022-01-28 15:09:59 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

2022-01-28 15:10:04 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 0.086 mins elapsed.

2022-01-28 15:10:04 : (rabbit_BGRGP1 : 1 of 

[1] 102


...

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR logging to : ArchRLogs/ArchR-createArrows-4e06111ceb15-Date-2022-01-28_Time-15-52-52.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:52:53 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:52:53 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:52:53 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 15:52:53 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

2022-01-28 15:52:58 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 0.086 mins elapsed.

2022-01-28 15:52:58 : (rabbit_BGRGP1 : 1 of 1) 

[1] 199


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR logging to : ArchRLogs/ArchR-createArrows-4e0638b2b38a-Date-2022-01-28_Time-15-53-21.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:53:22 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:53:22 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:53:22 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 15:53:22 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

2022-01-28 15:53:26 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 0.073 mins elapsed.

2022-01-28 15:53:26 : (rabbit_BGRGP1 : 1 of 1) 

[1] 200


Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..

ArchR logging to : ArchRLogs/ArchR-createArrows-4e06270bef62-Date-2022-01-28_Time-15-53-49.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:53:49 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-28 15:53:49 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:53:49 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 15:53:49 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

2022-01-28 15:53:54 : (rabbit_BGRGP1 : 1 of 1) Successful creation of Temporary File, 0.081 mins elapsed.

2022-01-28 15:53:54 : (rabbit_BGRGP1 : 1 of 1) 